<a href="https://colab.research.google.com/github/YenlingPeng/T-Brain_AI_esunbank/blob/Chiang/%E5%88%9D%E7%89%88%E8%A8%93%E7%B7%B4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

import torch
if torch.cuda.is_available():
    print("CUDA is available.")
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available. Please change the runtime type to GPU.")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns
# from tqdm.auto import tqdm
import time

Mounted at /content/drive
CUDA is not available. Please change the runtime type to GPU.


In [2]:
!wget -O TaipeiSansTCBeta-Regular.ttf https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_&export=download

import matplotlib

# 改style要在改font之前
# plt.style.use('seaborn')

matplotlib.font_manager.fontManager.addfont('TaipeiSansTCBeta-Regular.ttf')
matplotlib.rc('font', family='Taipei Sans TC Beta')

--2025-11-06 01:50:15--  https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_
Resolving drive.google.com (drive.google.com)... 173.194.206.139, 173.194.206.100, 173.194.206.101, ...
Connecting to drive.google.com (drive.google.com)|173.194.206.139|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_ [following]
--2025-11-06 01:50:15--  https://drive.usercontent.google.com/download?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.251.189.132, 2607:f8b0:4001:c6c::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.251.189.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 20659344 (20M) [application/octet-stream]
Saving to: ‘TaipeiSansTCBeta-Regular.ttf’

TaipeiSansTCBeta-Re 100%[===================>]  19.70M   126MB/s    in 0.2s    

2025-11-06 

In [3]:
# 載入資料集
transactions = '/content/drive/MyDrive/TBrainAI/40_初賽資料_V3 1/初賽資料/acct_transaction.csv' # 換成自己路徑
alert = '/content/drive/MyDrive/TBrainAI/40_初賽資料_V3 1/初賽資料/acct_alert.csv'
predict = '/content/drive/MyDrive/TBrainAI/40_初賽資料_V3 1/初賽資料/acct_predict.csv'
acct_transactions = pd.read_csv(transactions)
acct_alert = pd.read_csv(alert)
acct_predict = pd.read_csv(predict)

## Step 1：

* 主要幣別：先做 TWD

* 時段定義：夜間 20:00–06:00

* 交易通路分群：

        cash：1(ATM), 2(臨櫃), 6(eATM)

        wire：3(行銀), 4(網銀), 5(語音), 7(電子支付)

        其他/排除：99, UNK

In [4]:
acct_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4435890 entries, 0 to 4435889
Data columns (total 10 columns):
 #   Column          Dtype  
---  ------          -----  
 0   from_acct       object 
 1   from_acct_type  int64  
 2   to_acct         object 
 3   to_acct_type    int64  
 4   is_self_txn     object 
 5   txn_amt         float64
 6   txn_date        int64  
 7   txn_time        object 
 8   currency_type   object 
 9   channel_type    object 
dtypes: float64(1), int64(3), object(6)
memory usage: 338.4+ MB


In [5]:
acct_transactions.head()

,from_acct,from_acct_type,to_acct,to_acct_type,is_self_txn,txn_amt,txn_date,txn_time,currency_type,channel_type
0,be6fdd2d0f9aa02b0b09436fb137654942e3346e16ab43...,1,7abb16ac9bddc1f464981131ba68506775a964df2e0734...,1,N,47500.0,71,05:05:00,TWD,04
1,18f3d0e79217f8bc8b4cb485f9f80a884771b846de652f...,1,e77e425fb5f3ece7a7b431b3c43cc1d040f3054e35479d...,2,UNK,6150.0,31,20:55:00,TWD,03
2,302f3911cbf56bf9b5ad209a4b045a82380f98d92604c1...,1,4a707a0af2aa824777082803013610090033104c308023...,1,N,1150000.0,37,09:20:00,TWD,04
3,5a4809796865b1526f46e5dda6a35c1a4def3cbe969cc8...,1,d16b1bf33802f020b508002755c13aad549bc59dde7aae...,2,UNK,8550.0,106,13:40:00,TWD,04
4,7f84214987bdee16ffbaf3d70824e6385ce80e032a24c5...,1,c2e0f75b54f394b29755779ab9a488931e9d893a0e5f8f...,1,N,1450.0,84,11:20:00,TWD,03


In [6]:
# === Step 1. 讀取與清理 ===
df = acct_transactions.copy()
df = df[df["currency_type"] == "TWD"]
df = df[~df["channel_type"].isin(["99", "UNK"])]

# === Step 2. 通路分群 ===
cash_channels = {"1","2","6"}
wire_channels = {"3","4","5","7"}
def map_channel(x):
    x = str(x).lstrip("0")
    if x in cash_channels: return "cash"
    if x in wire_channels: return "wire"
    return "other"
df["txn_cat"] = df["channel_type"].apply(map_channel)

# === ✅ Step 2.5. 轉換時間 → 標記夜間 ===
df["txn_time"] = pd.to_datetime(df["txn_time"], format="%H:%M:%S", errors="coerce")
df["hour"] = df["txn_time"].dt.hour
df["is_night"] = ((df["hour"] >= 20) | (df["hour"] < 6)).astype(int)

# （之後 Step 3 開始建立帳戶方向）
df_in = df.rename(columns={"to_acct": "account_id"})
df_in["direction"] = "C"

df_out = df.rename(columns={"from_acct": "account_id"})
df_out["direction"] = "D"

acct_tx = pd.concat([df_in, df_out], ignore_index=True)

In [7]:
acct_tx.head()

,from_acct,from_acct_type,account_id,to_acct_type,is_self_txn,txn_amt,txn_date,txn_time,currency_type,channel_type,txn_cat,hour,is_night,direction,to_acct
0,be6fdd2d0f9aa02b0b09436fb137654942e3346e16ab43...,1,7abb16ac9bddc1f464981131ba68506775a964df2e0734...,1,N,47500.0,71,1900-01-01 05:05:00,TWD,04,wire,5,1,C,NaN
1,18f3d0e79217f8bc8b4cb485f9f80a884771b846de652f...,1,e77e425fb5f3ece7a7b431b3c43cc1d040f3054e35479d...,2,UNK,6150.0,31,1900-01-01 20:55:00,TWD,03,wire,20,1,C,NaN
2,302f3911cbf56bf9b5ad209a4b045a82380f98d92604c1...,1,4a707a0af2aa824777082803013610090033104c308023...,1,N,1150000.0,37,1900-01-01 09:20:00,TWD,04,wire,9,0,C,NaN
3,5a4809796865b1526f46e5dda6a35c1a4def3cbe969cc8...,1,d16b1bf33802f020b508002755c13aad549bc59dde7aae...,2,UNK,8550.0,106,1900-01-01 13:40:00,TWD,04,wire,13,0,C,NaN
4,7f84214987bdee16ffbaf3d70824e6385ce80e032a24c5...,1,c2e0f75b54f394b29755779ab9a488931e9d893a0e5f8f...,1,N,1450.0,84,1900-01-01 11:20:00,TWD,03,wire,11,0,C,NaN


---

##  Step 3～7：

**Step 3：基本帳戶統計**

* 將交易層資料彙整為帳戶層統計。
* 特徵：交易筆數、總金額、平均金額、波動度、活躍天數、夜間比例。
* 目的：建立帳戶行為的基礎活躍度與金流概況。

**Step 4：方向統計（C / D）**

* 區分匯入 (Credit, C) 與匯出 (Debit, D) 行為。
* 特徵：匯入／匯出筆數、金額總和、平均金額。
* 目的：分析帳戶是以收款還是付款為主。

**Step 5：通路方向統計（cash / wire）**

* 將交易依通路分類為「現金」與「匯款」。
* 特徵：cash_in/out、wire_in/out。
* 目的：觀察帳戶主要交易通路與資金型態。

**Step 6：金流比例特徵**

* 比較不同方向與通路間的金流比例。
* 特徵：cash_in_out_ratio、wire_in_out_ratio 等。
* 目的：偵測金流是否平衡（進出對等為潛在異常）。

**Step 7：每日與週期特徵**

* 將日期轉換為 weekday，分析交易週期性。
* 特徵：每日平均交易數／金額、weekday_0~6 比例。
* 目的：了解帳戶的時間行為模式與活動規律。


In [8]:
# =========================================
# 🏦 Account-level Feature Engineering (Fast + tqdm)
# =========================================

from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import time, gc

tqdm.pandas()

start_time = time.time()
print("🚀 開始帳戶層級特徵工程...")

# ========================================
# Step 3. 基本帳戶聚合統計
# ========================================
print("⏳ Step 3: 基本帳戶統計中...")
t1 = time.time()

base_agg = (
    acct_tx.groupby("account_id", observed=True)
           .agg(
               txn_count=("txn_amt","size"),
               amt_sum=("txn_amt","sum"),
               amt_mean=("txn_amt","mean"),
               amt_std=("txn_amt","std"),
               amt_min=("txn_amt","min"),
               amt_max=("txn_amt","max"),
               active_days=("txn_date", pd.Series.nunique),
               night_ratio=("is_night","mean")
           )
           .reset_index()
)
print(f"✅ Step 3 完成 ({time.time()-t1:.2f} 秒)")

# ========================================
# Step 4. 方向統計（C / D） → 改良版 groupby + unstack
# ========================================
print("⏳ Step 4: 方向統計 (C/D)...")
t1 = time.time()

dir_agg = (
    acct_tx.groupby(["account_id","direction"], observed=True)
           .agg(
               txn_cnt=("txn_amt","size"),
               amt_sum=("txn_amt","sum"),
               amt_mean=("txn_amt","mean"),
               amt_std=("txn_amt","std")
           )
           .unstack(fill_value=0)
)
dir_agg.columns = [f"{a}_{b}" for a,b in dir_agg.columns]
dir_agg = dir_agg.reset_index()

print(f"✅ Step 4 完成 ({time.time()-t1:.2f} 秒)")

# ========================================
# Step 5. 通路方向統計（cash_in / wire_out）
# ========================================
print("⏳ Step 5: 通路方向統計...")
t1 = time.time()

def channel_sum(cat, dir_):
    mask = (acct_tx["txn_cat"]==cat) & (acct_tx["direction"]==dir_)
    return acct_tx.loc[mask].groupby("account_id")["txn_amt"].sum()

cash_in = channel_sum("cash","C").rename("cash_in")
cash_out = channel_sum("cash","D").rename("cash_out")
wire_in = channel_sum("wire","C").rename("wire_in")
wire_out = channel_sum("wire","D").rename("wire_out")

flow_agg = (
    base_agg.merge(dir_agg, on="account_id", how="left")
            .merge(cash_in, on="account_id", how="left")
            .merge(cash_out, on="account_id", how="left")
            .merge(wire_in, on="account_id", how="left")
            .merge(wire_out, on="account_id", how="left")
            .fillna(0)
)
gc.collect()
print(f"✅ Step 5 完成 ({time.time()-t1:.2f} 秒)")

# ========================================
# Step 6. 金流比例特徵（不含 sigmoid）
# ========================================
print("⏳ Step 6: 計算金流比例特徵...")
t1 = time.time()

eps = 1e-9
flow_agg["cash_in_out_ratio"]  = flow_agg["cash_in"]  / (flow_agg["cash_out"] + eps)
flow_agg["wire_in_out_ratio"]  = flow_agg["wire_in"]  / (flow_agg["wire_out"] + eps)
flow_agg["cash_in_over_wire_out"] = flow_agg["cash_in"] / (flow_agg["wire_out"] + eps)
flow_agg["wire_in_over_cash_out"] = flow_agg["wire_in"] / (flow_agg["cash_out"] + eps)

print(f"✅ Step 6 完成 ({time.time()-t1:.2f} 秒)")

# ========================================
# Step 7. 每日活動統計（含 weekday 比例, 向量化加速）
# ========================================
print("⏳ Step 7: 計算每日與 weekday 特徵 (高速向量化)...")
t1 = time.time()

# 👉 你的 txn_date 是「以 1 為第一天」的編碼，假設第 1 天是星期一
acct_tx["txn_date"] = acct_tx["txn_date"].astype(int)
acct_tx["weekday"] = (acct_tx["txn_date"] - 1) % 7  # 0=Mon, 6=Sun

# 每帳戶每日聚合
per_day = (
    acct_tx.groupby(["account_id","txn_date"], observed=True)
           .agg(day_txn=("txn_amt","size"),
                day_sum=("txn_amt","sum"))
           .reset_index()
)

# 每帳戶每日統計
perday_stats = (
    per_day.groupby("account_id", observed=True)
           .agg(
               txn_per_day_mean=("day_txn","mean"),
               txn_per_day_std=("day_txn","std"),
               day_sum_mean=("day_sum","mean"),
               day_sum_std=("day_sum","std")
           )
           .reset_index()
)

# ✅ 向量化計算 weekday 比例
weekday_ratio = (
    acct_tx.groupby("account_id")["weekday"]
           .value_counts(normalize=True)
           .unstack(fill_value=0)
           .add_prefix("weekday_")
           .reset_index()
)

# 整合全部
feat = (
    flow_agg.merge(perday_stats, on="account_id", how="left")
            .merge(weekday_ratio, on="account_id", how="left")
            .fillna(0)
)
gc.collect()
print(f"✅ Step 7 完成 ({time.time()-t1:.2f} 秒)")

# ========================================
# Step 8. 完成輸出
# ========================================
end_time = time.time()
print(f"🏁 全部完成！共 {len(feat)} 個帳戶，用時 {end_time-start_time:.2f} 秒。")
print("🔹 欄位預覽：", feat.columns.tolist()[:15], "...")
display(feat.head(5))


🚀 開始帳戶層級特徵工程...
⏳ Step 3: 基本帳戶統計中...
✅ Step 3 完成 (85.20 秒)
⏳ Step 4: 方向統計 (C/D)...
✅ Step 4 完成 (9.95 秒)
⏳ Step 5: 通路方向統計...
✅ Step 5 完成 (18.25 秒)
⏳ Step 6: 計算金流比例特徵...
✅ Step 6 完成 (0.03 秒)
⏳ Step 7: 計算每日與 weekday 特徵 (高速向量化)...
✅ Step 7 完成 (31.32 秒)
🏁 全部完成！共 1197830 個帳戶，用時 144.75 秒。
🔹 欄位預覽： ['account_id', 'txn_count', 'amt_sum', 'amt_mean', 'amt_std', 'amt_min', 'amt_max', 'active_days', 'night_ratio', 'txn_cnt_C', 'txn_cnt_D', 'amt_sum_C', 'amt_sum_D', 'amt_mean_C', 'amt_mean_D'] ...


,account_id,txn_count,amt_sum,amt_mean,amt_std,amt_min,amt_max,active_days,night_ratio,txn_cnt_C,...,txn_per_day_std,day_sum_mean,day_sum_std,weekday_0,weekday_1,weekday_2,weekday_3,weekday_4,weekday_5,weekday_6
0,00000577cfcd0bde8ee693021419ef13a1f7f933ec8626...,1,4050.0,4050.0,0.000000,4050.0,4050.0,1,0.0,1,...,0.0,4050.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,00002846e6b430580825e2b10fe3ff1e3ddb93f42c608d...,1,3050.0,3050.0,0.000000,3050.0,3050.0,1,0.0,1,...,0.0,3050.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,00002b3d8f9c7b91c407a5725849deb521fcf1dd5eea1f...,1,75.0,75.0,0.000000,75.0,75.0,1,1.0,1,...,0.0,75.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0000319cf868e5a245bbe4726c9f8f0cbb7cbc03a9aa01...,2,71000.0,35500.0,21213.203436,20500.0,50500.0,1,0.0,1,...,0.0,71000.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,00003f3fb30775e809a2e02924f57f360f3d02e89cbd82...,1,3050.0,3050.0,0.000000,3050.0,3050.0,1,0.0,1,...,0.0,3050.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


### 帳戶行為有時候只有一筆資料，怕有些統計沒有意義，所以目前初步是想要分開訓練


*   設定閾值為3筆
    * 低於 3 筆 -> 筆數很少
    * 高於 3 筆 -> 筆數很多





In [9]:
MIN_TXN = 2  # 模型拆分門檻
feat_mad = feat[feat["txn_count"] >= MIN_TXN].copy() # 篩選後的資料筆數

print(f"✅ 保留 {len(feat_mad):,} 個帳戶 ({len(feat_mad)/len(feat)*100:.2f}%) 用於 模型訓練")

✅ 保留 394,323 個帳戶 (32.92%) 用於 模型訓練


In [12]:
# 假設這三個 DataFrame 都已經存在：
# feat_mad, acct_predict, acct_alert

# 確保欄位名稱一致
feat_ids = set(feat_mad["account_id"].unique())
predict_ids = set(acct_predict["acct"].unique())
alert_ids = set(acct_alert["acct"].unique())

# 各自的筆數
n_feat = len(feat_ids)
n_predict = len(predict_ids)
n_alert = len(alert_ids)

# 交集統計
predict_in_feat = len(predict_ids & feat_ids)
alert_in_feat = len(alert_ids & feat_ids)

print("📊 帳戶重疊統計報告")
print("──────────────────────────")
print(f"訓練用帳戶 (feat_mad)：{n_feat:,}")
print(f"預測帳戶 (acct_predict)：{n_predict:,} → 其中 {predict_in_feat:,} 筆在 feat_mad 中")
print(f"警示帳戶 (acct_alert)：{n_alert:,} → 其中 {alert_in_feat:,} 筆在 feat_mad 中")

# 若要比例顯示
print("──────────────────────────")
print(f"預測帳戶覆蓋率：{predict_in_feat / n_predict * 100:.2f}%")
print(f"警示帳戶覆蓋率：{alert_in_feat / n_alert * 100:.2f}%")


📊 帳戶重疊統計報告
──────────────────────────
訓練用帳戶 (feat_mad)：394,323
預測帳戶 (acct_predict)：4,780 → 其中 4,511 筆在 feat_mad 中
警示帳戶 (acct_alert)：1,004 → 其中 633 筆在 feat_mad 中
──────────────────────────
預測帳戶覆蓋率：94.37%
警示帳戶覆蓋率：63.05%


## 特徵工程
* 將帳戶層統計進一步轉換為具異常辨識力的特徵。
* **Sigmoid (`envelop_C_D`)：** 偵測進出金流是否對等（結構化帳戶）。
* **log 平滑：** 對金額類特徵取 log，降低極端值影響。
* **amt_cv：** 金額穩定度（變異係數）。
* **比例特徵平滑：** 平滑極端金流比。
* **weekday_entropy：** 觀察交易日分布是否集中。
* **Z-score 標準化：** 統一特徵尺度，便於異常模型輸入。

最終輸出為 `feat_final`，可用於帳戶異常偵測模型訓練。


In [11]:
from sklearn.preprocessing import StandardScaler

# ===============================================================
# MAD 進階特徵工程 (含 Sigmoid + log 平滑)
# ===============================================================
def enrich_mad_features(feat):
    feat = feat.copy()

    # -----------------------------------------------------------
    # Step 1️⃣ 結構化金流特徵（Sigmoid Envelop Function）
    # -----------------------------------------------------------
    def envelop_function(C, D, a=10, c1=0.8, c2=1.2):
        sigmoid = lambda x: 1 / (1 + np.exp(-x))
        ratio = np.divide(C, D, out=np.zeros_like(C), where=D>0)
        k = sigmoid(a*(c2 - c1)) - sigmoid(0)
        return np.where(
            D > 0,
            (1/k) * (sigmoid(a*(ratio - c1)) - sigmoid(a*(ratio - c2))),
            0
        )

    feat["envelop_C_D"] = envelop_function(feat["amt_sum_C"], feat["amt_sum_D"])

    # -----------------------------------------------------------
    # Step 2️⃣ log 平滑處理（針對金額型特徵）
    # -----------------------------------------------------------
    log_cols = ["amt_sum", "amt_mean", "amt_std"]
    for col in log_cols:
        if col in feat.columns:
            feat[f"log_{col}"] = np.log1p(feat[col].clip(lower=0))

    # -----------------------------------------------------------
    # Step 3️⃣ 金額波動與變異係數 (Coefficient of Variation)
    # -----------------------------------------------------------
    feat["amt_cv"] = (feat["amt_std"] / (feat["amt_mean"] + 1e-6)).clip(0, 10)

    # -----------------------------------------------------------
    # Step 4️⃣ 比例特徵平滑（避免極端值）
    # -----------------------------------------------------------
    ratio_cols = ["cash_in_out_ratio", "wire_in_out_ratio",
                  "cash_in_over_wire_out", "wire_in_over_cash_out"]
    for col in ratio_cols:
        if col in feat.columns:
            feat[col] = np.log1p(feat[col].clip(0, 1000))

    # -----------------------------------------------------------
    # Step 5️⃣ 週期行為特徵（weekday entropy）
    # -----------------------------------------------------------
    weekday_cols = [c for c in feat.columns if c.startswith("weekday_")]
    if weekday_cols:
        prob = feat[weekday_cols].values
        feat["weekday_entropy"] = -np.nansum(
            np.where(prob > 0, prob * np.log(prob), 0),
            axis=1
        )

    # -----------------------------------------------------------
    # Step 6️⃣ 標準化（Z-score）
    # -----------------------------------------------------------
    cols_to_scale = [
        "txn_count", "amt_sum", "amt_mean", "amt_std",
        "night_ratio", "cash_in_out_ratio", "wire_in_out_ratio",
        "envelop_C_D", "amt_cv", "weekday_entropy"
    ]
    cols_to_scale = [c for c in cols_to_scale if c in feat.columns]

    scaler = StandardScaler()
    feat[cols_to_scale] = scaler.fit_transform(feat[cols_to_scale])

    print(f"✅ MAD 特徵工程完成，共 {len(feat)} 筆帳戶、{len(cols_to_scale)} 個正規化欄位。")
    return feat

# ===============================================================
# 執行特徵工程（對所有帳戶）
# ===============================================================
feat_final = enrich_mad_features(feat)

# 預覽結果
display(feat_final.head())

/tmp/ipython-input-1548022099.py:53: RuntimeWarning: divide by zero encountered in log
  np.where(prob > 0, prob * np.log(prob), 0),
/tmp/ipython-input-1548022099.py:53: RuntimeWarning: invalid value encountered in multiply
  np.where(prob > 0, prob * np.log(prob), 0),


✅ MAD 特徵工程完成，共 1197830 筆帳戶、10 個正規化欄位。


,account_id,txn_count,amt_sum,amt_mean,amt_std,amt_min,amt_max,active_days,night_ratio,txn_cnt_C,...,weekday_3,weekday_4,weekday_5,weekday_6,envelop_C_D,log_amt_sum,log_amt_mean,log_amt_std,amt_cv,weekday_entropy
0,00000577cfcd0bde8ee693021419ef13a1f7f933ec8626...,-0.163195,-0.070909,-0.137420,-0.127137,4050.0,4050.0,1,-0.822222,1,...,0.0,0.0,0.0,0.0,-0.104790,8.306719,8.306719,0.000000,-0.492993,-0.605754
1,00002846e6b430580825e2b10fe3ff1e3ddb93f42c608d...,-0.163195,-0.071432,-0.146552,-0.127137,3050.0,3050.0,1,-0.822222,1,...,0.0,0.0,1.0,0.0,-0.104790,8.023225,8.023225,0.000000,-0.492993,-0.605754
2,00002b3d8f9c7b91c407a5725849deb521fcf1dd5eea1f...,-0.163195,-0.072989,-0.173721,-0.127137,75.0,75.0,1,1.461590,1,...,0.0,0.0,0.0,0.0,-0.104790,4.330733,4.330733,0.000000,-0.492993,-0.605754
3,0000319cf868e5a245bbe4726c9f8f0cbb7cbc03a9aa01...,-0.117393,-0.035867,0.149791,0.201254,20500.0,50500.0,1,-0.822222,1,...,0.0,0.0,0.0,0.0,-0.104736,11.170449,10.477316,9.962426,0.778134,-0.605754
4,00003f3fb30775e809a2e02924f57f360f3d02e89cbd82...,-0.163195,-0.071432,-0.146552,-0.127137,3050.0,3050.0,1,-0.822222,1,...,0.0,1.0,0.0,0.0,-0.104790,8.023225,8.023225,0.000000,-0.492993,-0.605754


## 訓練模型
 * 將模型分成 modelA 、modelB
  * 對 A 訓練 使用元學習


In [13]:
# ------------------------------------------------------------
# Step 1️⃣ 建立標籤
# ------------------------------------------------------------
alert_set = set(acct_alert["acct"])
feat_final["label"] = feat_final["account_id"].apply(
    lambda x: 1 if x in alert_set else 0
)
n_alert = feat_final["label"].sum()
n_total = len(feat_final)
print(f"📊 標記完成，共 {n_alert:,} 筆警示帳戶（{n_alert/n_total*100:.4f}%）")

# ------------------------------------------------------------
# Step 2️⃣ 分群：多筆帳戶 vs 少筆帳戶
# ------------------------------------------------------------
feat_modelA = feat_final[feat_final["txn_count"] >= MIN_TXN].copy()
feat_modelB = feat_final[feat_final["txn_count"] < MIN_TXN].copy()

print(f"Model A（多筆帳戶）：{len(feat_modelA):,} 筆")
print(f"Model B（少筆帳戶）：{len(feat_modelB):,} 筆")

# 各組警示帳戶比例
alert_A = feat_modelA["label"].sum()
alert_B = feat_modelB["label"].sum()
print(f" ├─ Model A 警示帳戶：{alert_A:,} 筆（{alert_A/len(feat_modelA)*100:.4f}%）")
print(f" └─ Model B 警示帳戶：{alert_B:,} 筆（{alert_B/len(feat_modelB)*100:.4f}%）")

# ------------------------------------------------------------
# Step 3️⃣ 建立訓練集與預測集
# ------------------------------------------------------------
predict_set = set(acct_predict["acct"])

# 多筆帳戶
trainA = feat_modelA[~feat_modelA["account_id"].isin(predict_set)]
predA  = feat_modelA[feat_modelA["account_id"].isin(predict_set)]

# 少筆帳戶
trainB = feat_modelB[~feat_modelB["account_id"].isin(predict_set)]
predB  = feat_modelB[feat_modelB["account_id"].isin(predict_set)]

# 訓練/預測數量統計
n_train = len(trainA) + len(trainB)
n_pred  = len(predA) + len(predB)
print(f"訓練資料：{n_train:,} 筆（A:{len(trainA):,}，B:{len(trainB):,}）")
print(f"預測資料：{n_pred:,} 筆（A:{len(predA):,}，B:{len(predB):,}）")

# 訓練集的警示比例
alert_trainA = trainA["label"].sum()
alert_trainB = trainB["label"].sum()
print(f" ├─ Train A 警示佔比：{alert_trainA/len(trainA)*100:.4f}%")
print(f" └─ Train B 警示佔比：{alert_trainB/len(trainB)*100:.4f}%")


📊 標記完成，共 785 筆警示帳戶（0.0655%）
Model A（多筆帳戶）：18,027 筆
Model B（少筆帳戶）：1,179,803 筆
 ├─ Model A 警示帳戶：63 筆（0.3495%）
 └─ Model B 警示帳戶：722 筆（0.0612%）
訓練資料：1,193,255 筆（A:16,734，B:1,176,521）
預測資料：4,575 筆（A:1,293，B:3,282）
 ├─ Train A 警示佔比：0.3765%
 └─ Train B 警示佔比：0.0614%


### 對 modelA 先進行模型訓練

In [15]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score
from imblearn.over_sampling import ADASYN
from lightgbm import LGBMClassifier

In [17]:
def train_with_adasyn_ratio(model, X, y, n_splits=3, target_ratio=0.25):
    """
    使用 ADASYN 生成樣本（只在訓練集內）
    target_ratio = 希望「正樣本 / 總樣本」的比例，例如 0.25 表示 2:8
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    f1s, pres, recs = [], [], []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n===== Fold {fold} =====")
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        pos = (y_train == 1).sum()
        neg = (y_train == 0).sum()
        current_ratio = pos / (pos + neg)

        # 希望最終達到 target_ratio，反推需要多少正樣本
        desired_pos = int(target_ratio * (pos + neg) / (1 - target_ratio))
        n_generate = max(desired_pos - pos, 0)
        sampling_strategy = (pos + n_generate) / neg

        print(f"✅ 原始: 正={pos}, 負={neg} → 目標比例 1:{(1-target_ratio)/target_ratio:.1f}")
        print(f"➡️  生成 {n_generate} 筆，sampling_strategy={sampling_strategy:.3f}")

        # 只在訓練集進行 ADASYN
        ada = ADASYN(sampling_strategy=sampling_strategy, random_state=42, n_neighbors=3)
        X_res, y_res = ada.fit_resample(X_train, y_train)
        print(f"生成後: 正樣本 {sum(y_res==1)}, 負樣本 {sum(y_res==0)}")

        # 訓練模型
        model.fit(X_res, y_res)

        # 驗證
        y_pred = model.predict(X_valid)
        f1 = f1_score(y_valid, y_pred)
        pre = precision_score(y_valid, y_pred, zero_division=0)
        rec = recall_score(y_valid, y_pred)

        f1s.append(f1); pres.append(pre); recs.append(rec)
        print(f"Fold {fold}: F1={f1:.4f}, Precision={pre:.4f}, Recall={rec:.4f}")

    print(f"\n📊 平均 F1={np.mean(f1s):.4f} ±{np.std(f1s):.4f}")
    print(f"平均 Precision={np.mean(pres):.4f}, Recall={np.mean(recs):.4f}")
    return np.mean(f1s)


In [18]:
drop_cols = ["account_id", "label"]
X_train_A = trainA.drop(columns=drop_cols, errors='ignore')
y_train_A = trainA["label"]

lgbm_model = LGBMClassifier(
    n_estimators=400, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    class_weight='balanced',   # ✅ 仍可使用
    random_state=42
)

train_with_adasyn_ratio(lgbm_model, X_train_A, y_train_A, n_splits=3, target_ratio=0.2)



===== Fold 1 =====
✅ 原始: 正=42, 負=11114 → 目標比例 1:4.0
➡️  生成 2747 筆，sampling_strategy=0.251
生成後: 正樣本 2772, 負樣本 11114
[LightGBM] [Info] Number of positive: 2772, number of negative: 11114
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006289 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10295
[LightGBM] [Info] Number of data points in the train set: 13886, number of used features: 41
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Fold 1: F1=0.0000, Precision=0.0000, Recall=0.0000

===== Fold 2 =====
✅ 原始: 正=42, 負=11114 → 目標比例 1:4.0
➡️  生成 2747 筆，sampling_strategy=0.251
生成後: 正樣本 2770, 負樣本 11114
[LightGBM] [Info] Number of positive: 2770, number of negative: 11114
[Li

np.float64(0.0)

In [19]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score
from imblearn.over_sampling import ADASYN
from xgboost import XGBClassifier
import numpy as np

def train_with_adasyn_xgb(X, y, n_splits=3, target_ratio=0.2):
    """
    使用 ADASYN + XGBoost (控制生成比例)
    target_ratio = 希望「正樣本 / 總樣本」的比例 (例如 0.2 表示正樣本佔 20%)
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    f1s, pres, recs = [], [], []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n===== Fold {fold} =====")
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        # 原始比例
        pos = (y_train == 1).sum()
        neg = (y_train == 0).sum()
        print(f"✅ 原始: 正={pos}, 負={neg}")

        # 控制 ADASYN 生成比例
        desired_pos = int(target_ratio * (pos + neg) / (1 - target_ratio))
        n_generate = max(desired_pos - pos, 0)
        sampling_strategy = (pos + n_generate) / neg

        ada = ADASYN(sampling_strategy=sampling_strategy, random_state=42, n_neighbors=3)
        X_res, y_res = ada.fit_resample(X_train, y_train)
        print(f"➡️  生成後: 正樣本 {sum(y_res==1)}, 負樣本 {sum(y_res==0)}")

        # 計算 scale_pos_weight（負樣本 / 正樣本）
        scale_pos = sum(y_res == 0) / sum(y_res == 1)
        print(f"⚖️  scale_pos_weight = {scale_pos:.3f}")

        # 建立 XGBoost 模型
        model = XGBClassifier(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos,  # ✅ 根據重抽樣後比例設定
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
            use_label_encoder=False
        )

        model.fit(X_res, y_res)

        # 驗證
        y_pred = model.predict(X_valid)
        f1 = f1_score(y_valid, y_pred)
        pre = precision_score(y_valid, y_pred, zero_division=0)
        rec = recall_score(y_valid, y_pred)

        f1s.append(f1)
        pres.append(pre)
        recs.append(rec)
        print(f"Fold {fold}: F1={f1:.4f}, Precision={pre:.4f}, Recall={rec:.4f}")

    print(f"\n📊 平均 F1={np.mean(f1s):.4f} ±{np.std(f1s):.4f}")
    print(f"平均 Precision={np.mean(pres):.4f}, Recall={np.mean(recs):.4f}")
    return np.mean(f1s)

In [20]:
drop_cols = ["account_id", "label"]
X_train_A = trainA.drop(columns=drop_cols, errors='ignore')
y_train_A = trainA["label"]

train_with_adasyn_xgb(X_train_A, y_train_A, n_splits=3, target_ratio=0.2)



===== Fold 1 =====
✅ 原始: 正=42, 負=11114
➡️  生成後: 正樣本 2772, 負樣本 11114
⚖️  scale_pos_weight = 4.009


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [02:18:12] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fold 1: F1=0.0000, Precision=0.0000, Recall=0.0000

===== Fold 2 =====
✅ 原始: 正=42, 負=11114
➡️  生成後: 正樣本 2770, 負樣本 11114
⚖️  scale_pos_weight = 4.012


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [02:18:18] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fold 2: F1=0.0000, Precision=0.0000, Recall=0.0000

===== Fold 3 =====
✅ 原始: 正=42, 負=11114
➡️  生成後: 正樣本 2792, 負樣本 11114
⚖️  scale_pos_weight = 3.981


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [02:18:21] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fold 3: F1=0.0000, Precision=0.0000, Recall=0.0000

📊 平均 F1=0.0000 ±0.0000
平均 Precision=0.0000, Recall=0.0000


np.float64(0.0)

In [21]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_curve
from imblearn.over_sampling import RandomOverSampler
from xgboost import XGBClassifier
import numpy as np

def train_with_ros_xgb_threshold(X, y, n_splits=3, target_ratio=0.2):
    """
    XGBoost + RandomOverSampler (控制比例) + threshold 掃描
    target_ratio = 希望「正樣本 / 總樣本」的比例 (例如 0.2 表示正樣本佔 20%)
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    f1s, pres, recs, aucs, thresholds = [], [], [], [], []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n===== Fold {fold} =====")
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        pos = (y_train == 1).sum()
        neg = (y_train == 0).sum()
        print(f"✅ 原始: 正={pos}, 負={neg}")

        # ⚖️ 隨機過採樣到目標比例
        ros = RandomOverSampler(sampling_strategy=target_ratio, random_state=42)
        X_res, y_res = ros.fit_resample(X_train, y_train)
        print(f"➡️  過採樣後: 正樣本 {sum(y_res==1)}, 負樣本 {sum(y_res==0)}")

        # 建立 XGBoost
        scale_pos = sum(y_res==0) / sum(y_res==1)
        model = XGBClassifier(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos,  # 強化正樣本損失
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )

        # 訓練
        model.fit(X_res, y_res)

        # 預測機率
        y_prob = model.predict_proba(X_valid)[:,1]

        # 掃描最佳 threshold (maximize F1)
        precisions, recalls, thrs = precision_recall_curve(y_valid, y_prob)
        f1s_curve = 2 * precisions * recalls / (precisions + recalls + 1e-9)
        best_idx = np.argmax(f1s_curve)
        best_thr = thrs[best_idx]
        best_f1, best_pre, best_rec = f1s_curve[best_idx], precisions[best_idx], recalls[best_idx]

        print(f"🎯 最佳 F1={best_f1:.4f}, Precision={best_pre:.4f}, Recall={best_rec:.4f}, threshold={best_thr:.3f}")

        f1s.append(best_f1)
        pres.append(best_pre)
        recs.append(best_rec)
        thresholds.append(best_thr)

    # 平均結果
    print("\n📊 === K-Fold 平均表現 ===")
    print(f"平均 F1={np.mean(f1s):.4f} ±{np.std(f1s):.4f}")
    print(f"平均 Precision={np.mean(pres):.4f}, Recall={np.mean(recs):.4f}")
    print(f"平均最佳 threshold={np.mean(thresholds):.3f}")

    return {
        "f1_mean": np.mean(f1s),
        "precision_mean": np.mean(pres),
        "recall_mean": np.mean(recs),
        "threshold_mean": np.mean(thresholds)
    }


In [22]:
drop_cols = ["account_id", "label"]
X_train_A = trainA.drop(columns=drop_cols, errors='ignore')
y_train_A = trainA["label"]

res = train_with_ros_xgb_threshold(X_train_A, y_train_A, n_splits=3, target_ratio=0.2)


===== Fold 1 =====
✅ 原始: 正=42, 負=11114
➡️  過採樣後: 正樣本 2222, 負樣本 11114
🎯 最佳 F1=0.0197, Precision=0.0106, Recall=0.1429, threshold=0.003

===== Fold 2 =====
✅ 原始: 正=42, 負=11114
➡️  過採樣後: 正樣本 2222, 負樣本 11114
🎯 最佳 F1=0.0274, Precision=0.0192, Recall=0.0476, threshold=0.017

===== Fold 3 =====
✅ 原始: 正=42, 負=11114
➡️  過採樣後: 正樣本 2222, 負樣本 11114
🎯 最佳 F1=0.0488, Precision=0.0500, Recall=0.0476, threshold=0.040

📊 === K-Fold 平均表現 ===
平均 F1=0.0320 ±0.0123
平均 Precision=0.0266, Recall=0.0794
平均最佳 threshold=0.020
